# OpenCV-Based System for Crack Identification and Width Estimation
## Fixed Width Measurement, Crack-wise Report Assignment and Risk Classification


In [ ]:
!pip -q install opencv-python scikit-image pandas matplotlib pillow
import os, cv2, zipfile, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from skimage.morphology import skeletonize


In [ ]:
PROJECT_NAME = "OpenCV-Based System for Crack Identification and Width Estimation"
OUTPUT_DIR="crack_results"; ANNOTATED_DIR=os.path.join(OUTPUT_DIR,"annotated_images"); MASK_DIR=os.path.join(OUTPUT_DIR,"crack_masks"); PROCESSED_DIR=os.path.join(OUTPUT_DIR,"processed_images"); REPORT_DIR=os.path.join(OUTPUT_DIR,"reports")
MAX_IMAGE_SIZE=1600; GAUSSIAN_KERNEL=5; CLAHE_CLIP_LIMIT=2.0; CLAHE_TILE_GRID=(8,8); BLACKHAT_KERNEL=21; MORPH_KERNEL_SIZE=3; MORPH_CLOSE_ITERATIONS=2; MORPH_OPEN_ITERATIONS=1; MIN_CRACK_AREA=20; MEASUREMENT_INTERVAL=20
CALIBRATION_ENABLED=True; ORIGINAL_PIXELS_PER_MM=10.0; GOOD_THRESHOLD_MM=0.30; MEDIUM_THRESHOLD_MM=1.00; MIN_WIDTH_MM=0.02; RISK_PERCENTILE=95; MAX_DISPLAY_POINTS=40
for d in [OUTPUT_DIR,ANNOTATED_DIR,MASK_DIR,PROCESSED_DIR,REPORT_DIR]: os.makedirs(d,exist_ok=True)
print(f"Calibration: {CALIBRATION_ENABLED}"); print(f"Original pixels/mm: {ORIGINAL_PIXELS_PER_MM}"); print("Risk: NORMAL <= 0.30 mm | MEDIUM > 0.30 to 1.00 mm | HIGH > 1.00 mm")


In [ ]:
ALLOWED_EXTENSIONS={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
uploaded=files.upload()
image_paths=[f for f in uploaded if os.path.splitext(f)[1].lower() in ALLOWED_EXTENSIONS]
if not image_paths: raise RuntimeError('No supported image files were uploaded.')


In [ ]:
def read_image(path):
    image=cv2.imread(path)
    if image is None or image.size==0: raise ValueError(f'Cannot read image: {path}')
    return image

def resize_image(image):
    h,w=image.shape[:2]; largest=max(h,w)
    if largest<=MAX_IMAGE_SIZE: return image.copy(),1.0
    scale=MAX_IMAGE_SIZE/float(largest); nw=max(1,int(round(w*scale))); nh=max(1,int(round(h*scale)))
    return cv2.resize(image,(nw,nh),interpolation=cv2.INTER_AREA),scale

def preprocess(image):
    gray=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY); k=GAUSSIAN_KERNEL+(GAUSSIAN_KERNEL%2==0); blurred=cv2.GaussianBlur(gray,(k,k),0)
    clahe=cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT,tileGridSize=CLAHE_TILE_GRID).apply(blurred)
    bk=BLACKHAT_KERNEL+(BLACKHAT_KERNEL%2==0); bh=cv2.morphologyEx(clahe,cv2.MORPH_BLACKHAT,cv2.getStructuringElement(cv2.MORPH_RECT,(bk,bk)))
    _,binary=cv2.threshold(bh,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU); mk=max(1,MORPH_KERNEL_SIZE); kernel=np.ones((mk,mk),np.uint8)
    cleaned=cv2.morphologyEx(binary,cv2.MORPH_OPEN,kernel,iterations=MORPH_OPEN_ITERATIONS); cleaned=cv2.morphologyEx(cleaned,cv2.MORPH_CLOSE,kernel,iterations=MORPH_CLOSE_ITERATIONS)
    n,labels,stats,_=cv2.connectedComponentsWithStats(cleaned,8); segmented=np.zeros_like(cleaned)
    for lab in range(1,n):
        if stats[lab,cv2.CC_STAT_AREA]>=MIN_CRACK_AREA: segmented[labels==lab]=255
    return gray,blurred,clahe,bh,binary,cleaned,segmented


In [ ]:
def estimate_crack_width(skeleton,distance_map,ppm):
    rows,cols=np.where(skeleton); out=[]
    for pid,(y,x) in enumerate(zip(rows,cols),1):
        d=float(distance_map[y,x])
        if d<=0: continue
        wpx=2.0*d; wmm=wpx/ppm if CALIBRATION_ENABLED and ppm>0 else np.nan
        if CALIBRATION_ENABLED and wmm<MIN_WIDTH_MM: continue
        out.append({'Point_ID':pid,'X':int(x),'Y':int(y),'Distance_Pixels':d,'Width_Pixels':wpx,'Width_MM':wmm})
    df=pd.DataFrame(out)
    if df.empty: return df
    return df

def classify_severity(width_mm):
    if not CALIBRATION_ENABLED or not np.isfinite(width_mm): return 'N/A','N/A'
    if width_mm<=GOOD_THRESHOLD_MM: return 'NORMAL','LOW'
    if width_mm<=MEDIUM_THRESHOLD_MM: return 'MEDIUM','MEDIUM'
    return 'HIGH','HIGH'

def assign_crack_ids(df,segmented):
    if df.empty: return df
    labels=cv2.connectedComponents((segmented>0).astype(np.uint8),8)[1]; x=df['X'].to_numpy(); y=df['Y'].to_numpy(); result=df.copy()
    result['Crack_ID']=[int(labels[int(yy),int(xx)]) for xx,yy in zip(x,y)]; return result[result['Crack_ID']>0].copy()

def crack_wise_report(df):
    cols=['Crack_ID','Measurement_Points','Minimum_Width_MM','Maximum_Width_MM','Average_Width_MM','Median_Width_MM','P95_Width_MM','Severity','Risk_Level']
    if df.empty: return pd.DataFrame(columns=cols)
    rows=[]
    for cid,g in df.groupby('Crack_ID',sort=True):
        a=pd.to_numeric(g['Width_MM'],errors='coerce').dropna().to_numpy(float)
        p95=float(np.percentile(a,RISK_PERCENTILE))
        sev,risk=classify_severity(p95)
        rows.append({'Crack_ID':int(cid),'Measurement_Points':len(g),'Minimum_Width_MM':float(a.min()),'Maximum_Width_MM':float(a.max()),'Average_Width_MM':float(a.mean()),'Median_Width_MM':float(np.median(a)),'P95_Width_MM':p95,'Severity':sev,'Risk_Level':risk})
    return pd.DataFrame(rows,columns=cols)


In [ ]:
def process_single_image(path):
    filename=os.path.basename(path); base=os.path.splitext(filename)[0]; original=read_image(path); resized,scale=resize_image(original); ppm=ORIGINAL_PIXELS_PER_MM*scale
    gray,blurred,clahe,bh,binary,cleaned,segmented=preprocess(resized); contours,_=cv2.findContours(segmented,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE); contours=[c for c in contours if cv2.contourArea(c)>=MIN_CRACK_AREA]
    skeleton=skeletonize(segmented>0); distance_map=cv2.distanceTransform(segmented,cv2.DIST_L2,5); measurements=estimate_crack_width(skeleton,distance_map,ppm); measurements=assign_crack_ids(measurements,segmented); crack_report=crack_wise_report(measurements)
    all_mm=measurements['Width_MM'].to_numpy(float) if not measurements.empty else np.array([]); risk_width=float(np.percentile(all_mm,RISK_PERCENTILE)) if all_mm.size else np.nan; severity,risk=classify_severity(risk_width)
    area_px=float(sum(cv2.contourArea(c) for c in contours)); area_mm2=area_px/(ppm**2) if ppm>0 else np.nan; length_px=float(np.count_nonzero(skeleton)); length_mm=length_px/ppm if ppm>0 else np.nan
    annotated=resized.copy(); cv2.drawContours(annotated,contours,-1,(255,0,255),2); annotated[skeleton]=(0,255,0)
    show_df=measurements.iloc[::max(1,MEASUREMENT_INTERVAL)].copy() if not measurements.empty else measurements
    if not measurements.empty:
        mx=measurements['Width_Pixels'].idxmax();
        if mx not in show_df.index: show_df=pd.concat([show_df,measurements.loc[[mx]]])
    show_df=show_df.drop_duplicates(subset=['Point_ID']).head(MAX_DISPLAY_POINTS)
    for _,r in show_df.iterrows():
        x,y=int(r['X']),int(r['Y']); cv2.circle(annotated,(x,y),4,(0,165,255),-1); cv2.putText(annotated,f"{r['Width_MM']:.2f} mm",(min(x+6,annotated.shape[1]-120),max(18,y-6)),cv2.FONT_HERSHEY_SIMPLEX,0.45,(0,165,255),1,cv2.LINE_AA)
    for i,t in enumerate([f'Cracks: {len(crack_report)}',f'P95 Width: {risk_width:.3f} mm' if np.isfinite(risk_width) else 'P95 Width: N/A',f'Severity: {severity}',f'Risk: {risk}']): cv2.putText(annotated,t,(12,28+i*28),cv2.FONT_HERSHEY_SIMPLEX,0.65,(255,255,255),2,cv2.LINE_AA)
    cv2.imwrite(os.path.join(MASK_DIR,f'{base}_mask.png'),segmented); cv2.imwrite(os.path.join(PROCESSED_DIR,f'{base}_skeleton.png'),skeleton.astype(np.uint8)*255); cv2.imwrite(os.path.join(PROCESSED_DIR,f'{base}_distance_transform.png'),cv2.normalize(distance_map,None,0,255,cv2.NORM_MINMAX).astype(np.uint8)); cimg=resized.copy(); cv2.drawContours(cimg,contours,-1,(0,255,0),2); cv2.imwrite(os.path.join(PROCESSED_DIR,f'{base}_contours.png'),cimg); final_path=os.path.join(ANNOTATED_DIR,f'{base}_annotated.png'); cv2.imwrite(final_path,annotated)
    if not measurements.empty: measurements.insert(0,'Image_Name',filename)
    result={'Image_Name':filename,'Image_Width':resized.shape[1],'Image_Height':resized.shape[0],'Original_Width':original.shape[1],'Original_Height':original.shape[0],'Resize_Scale':scale,'Crack_Detected':'YES' if not crack_report.empty else 'NO','Crack_Regions':len(crack_report),'Crack_Area_Pixels':area_px,'Crack_Area_MM2':area_mm2,'Crack_Length_Pixels':length_px,'Crack_Length_MM':length_mm,'Minimum_Width_MM':float(all_mm.min()) if all_mm.size else np.nan,'Maximum_Width_MM':float(all_mm.max()) if all_mm.size else np.nan,'Average_Width_MM':float(all_mm.mean()) if all_mm.size else np.nan,'Median_Width_MM':float(np.median(all_mm)) if all_mm.size else np.nan,'P95_Width_MM':risk_width,'Risk_Width_MM':risk_width,'Calibration_Enabled':CALIBRATION_ENABLED,'Pixels_Per_MM_Resized':ppm,'Severity':severity,'Risk_Level':risk,'Health_Status':severity if not crack_report.empty else 'NO CRACK DETECTED','Automatic_Message':f'Risk classification uses P{RISK_PERCENTILE} width: {risk_width:.3f} mm.' if np.isfinite(risk_width) else 'No measurable crack width.','Measurement_Points':len(measurements),'Annotated_Path':final_path}
    return result,measurements,crack_report


In [ ]:
results=[]; detailed=[]; crackwise=[]; failed=[]
for p in image_paths:
    try:
        r,m,c=process_single_image(p); results.append(r)
        if not m.empty: detailed.append(m)
        if not c.empty: c.insert(0,'Image_Name',os.path.basename(p)); crackwise.append(c)
    except Exception as e:
        failed.append({'filename':os.path.basename(p),'error':str(e)}); traceback.print_exc()
summary_df=pd.DataFrame(results); detailed_df=pd.concat(detailed,ignore_index=True) if detailed else pd.DataFrame(); crack_wise_df=pd.concat(crackwise,ignore_index=True) if crackwise else pd.DataFrame()
summary_df.to_csv(os.path.join(REPORT_DIR,'crack_analysis_report.csv'),index=False); detailed_df.to_csv(os.path.join(REPORT_DIR,'crack_measurement_points.csv'),index=False); crack_wise_df.to_csv(os.path.join(REPORT_DIR,'crack_wise_report.csv'),index=False); pd.DataFrame(failed,columns=['filename','error']).to_csv(os.path.join(REPORT_DIR,'processing_errors.csv'),index=False)
display(summary_df.round(4)); display(crack_wise_df.round(4))


In [ ]:
ZIP_PATH='crack_results.zip'
if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as z:
    for root,_,files_in_root in os.walk(OUTPUT_DIR):
        for f in files_in_root:
            fp=os.path.join(root,f); z.write(fp,os.path.relpath(fp,os.path.dirname(OUTPUT_DIR)))
print(f'Created: {ZIP_PATH}')
files.download(ZIP_PATH)
